# Model Training - LSTM Baseline & TCN

This notebook implements and trains:
1. **LSTM Baseline** - Standard recurrent neural network for comparison
2. **TCN (Temporal Convolutional Network)** - Main model for weather prediction

Both models predict temperature 24 hours ahead using 7 days of historical data.

In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

Using device: cpu


## 1. Configuration

In [2]:
# Hyperparameters - OPTIMIZED
CONFIG = {
    # Data - Shorter horizon = much better accuracy
    'sequence_length': 168,      # 7 days of hourly data
    'forecast_horizon': 6,        # Predict 6 hours ahead (not 24!)
    'train_ratio': 0.7,
    'val_ratio': 0.15,
    'target_col': 0,
    
    # Training
    'batch_size': 128,           # Larger batch for stability
    'epochs': 100,               # More epochs
    'learning_rate': 0.0005,     # Lower learning rate (important!)
    'early_stopping_patience': 15,
    
    # LSTM Model
    'lstm_hidden_size': 128,     # Larger
    'lstm_num_layers': 2,
    'lstm_dropout': 0.1,         # Less dropout
    
    # TCN Model  
    'tcn_num_channels': [64, 128, 128, 64],  # Better architecture
    'tcn_kernel_size': 7,         # Larger kernel
    'tcn_dropout': 0.1,
}

## 2. Load and Prepare Data

In [3]:
# Load the Jena Climate dataset
DATA_PATH = '../data/raw/jena_climate_2009_2016.csv'

df = pd.read_csv(DATA_PATH)
df['Date Time'] = pd.to_datetime(df['Date Time'], format='%d.%m.%Y %H:%M:%S')
df.set_index('Date Time', inplace=True)

# Resample to hourly (using lowercase 'h' for pandas 2.0+)
df_hourly = df.resample('h').mean()
df_hourly = df_hourly.interpolate(method='linear')

print(f'Dataset shape: {df_hourly.shape}')
print(f'Date range: {df_hourly.index.min()} to {df_hourly.index.max()}')

Dataset shape: (70129, 14)
Date range: 2009-01-01 00:00:00 to 2017-01-01 00:00:00


In [4]:
# Select features for modeling
feature_cols = ['T (degC)', 'p (mbar)', 'rh (%)', 'wv (m/s)', 
                'Tdew (degC)', 'VPdef (mbar)', 'rho (g/m**3)', 'sh (g/kg)']

data = df_hourly[feature_cols].values
print(f'Selected {len(feature_cols)} features')
print(f'Total samples: {len(data)}')

Selected 8 features
Total samples: 70129


In [5]:
# Train/Val/Test split (chronological)
n = len(data)
train_end = int(n * CONFIG['train_ratio'])
val_end = int(n * (CONFIG['train_ratio'] + CONFIG['val_ratio']))

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

print(f'Training samples:   {len(train_data):,}')
print(f'Validation samples: {len(val_data):,}')
print(f'Test samples:       {len(test_data):,}')

Training samples:   49,090
Validation samples: 10,519
Test samples:       10,520


In [6]:
# Normalize data using training statistics
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_data)
val_scaled = scaler.transform(val_data)
test_scaled = scaler.transform(test_data)

# Save scaler for later use
os.makedirs('../outputs/models', exist_ok=True)
with open('../outputs/models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('Data normalized and scaler saved!')

Data normalized and scaler saved!


## 3. Create PyTorch Dataset

In [7]:
class WeatherDataset(Dataset):
    """PyTorch Dataset for weather time series forecasting."""
    
    def __init__(self, data, sequence_length, forecast_horizon, target_col=0):
        self.data = torch.FloatTensor(data)
        self.sequence_length = sequence_length
        self.forecast_horizon = forecast_horizon
        self.target_col = target_col
        
    def __len__(self):
        return len(self.data) - self.sequence_length - self.forecast_horizon + 1
    
    def __getitem__(self, idx):
        # Input: sequence_length timesteps, all features
        x = self.data[idx:idx + self.sequence_length]
        
        # Target: forecast_horizon future values of target column
        y_start = idx + self.sequence_length
        y_end = y_start + self.forecast_horizon
        y = self.data[y_start:y_end, self.target_col]
        
        return x, y

In [8]:
# Create datasets and dataloaders
train_dataset = WeatherDataset(train_scaled, CONFIG['sequence_length'], 
                               CONFIG['forecast_horizon'], CONFIG['target_col'])
val_dataset = WeatherDataset(val_scaled, CONFIG['sequence_length'], 
                             CONFIG['forecast_horizon'], CONFIG['target_col'])
test_dataset = WeatherDataset(test_scaled, CONFIG['sequence_length'], 
                              CONFIG['forecast_horizon'], CONFIG['target_col'])

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print(f'Training batches:   {len(train_loader)}')
print(f'Validation batches: {len(val_loader)}')
print(f'Test batches:       {len(test_loader)}')

# Verify shapes
for x, y in train_loader:
    print(f'\nInput shape:  {x.shape}  (batch, sequence, features)')
    print(f'Target shape: {y.shape}  (batch, forecast_horizon)')
    break

Training batches:   383
Validation batches: 81
Test batches:       81

Input shape:  torch.Size([128, 168, 8])  (batch, sequence, features)
Target shape: torch.Size([128, 6])  (batch, forecast_horizon)


## 4. LSTM Baseline Model

In [9]:
class LSTMModel(nn.Module):
    """LSTM-based baseline model for time series forecasting."""
    
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.2):
        super(LSTMModel, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        # x shape: (batch, sequence, features)
        lstm_out, _ = self.lstm(x)
        
        # Use last timestep's output
        last_output = lstm_out[:, -1, :]
        out = self.dropout(last_output)
        out = self.fc(out)
        
        return out

In [10]:
# Create LSTM model
input_size = len(feature_cols)
output_size = CONFIG['forecast_horizon']

lstm_model = LSTMModel(
    input_size=input_size,
    hidden_size=CONFIG['lstm_hidden_size'],
    num_layers=CONFIG['lstm_num_layers'],
    output_size=output_size,
    dropout=CONFIG['lstm_dropout']
).to(device)

print(lstm_model)
print(f'\nTotal parameters: {sum(p.numel() for p in lstm_model.parameters()):,}')

LSTMModel(
  (lstm): LSTM(8, 128, num_layers=2, batch_first=True, dropout=0.1)
  (dropout): Dropout(p=0.1, inplace=False)
  (fc): Linear(in_features=128, out_features=6, bias=True)
)

Total parameters: 203,526


## 5. TCN Model

In [11]:
class CausalConv1d(nn.Module):
    """Causal 1D convolution - ensures no future information leakage."""
    
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super(CausalConv1d, self).__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels, out_channels, kernel_size,
            padding=self.padding, dilation=dilation
        )
        
    def forward(self, x):
        out = self.conv(x)
        # Remove future timesteps to maintain causality
        return out[:, :, :-self.padding] if self.padding > 0 else out


class TemporalBlock(nn.Module):
    """Temporal Block with dilated causal convolutions and residual connection."""
    
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.2):
        super(TemporalBlock, self).__init__()
        
        # First convolution
        self.conv1 = CausalConv1d(in_channels, out_channels, kernel_size, dilation)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        
        # Second convolution
        self.conv2 = CausalConv1d(out_channels, out_channels, kernel_size, dilation)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        
        # Residual connection (1x1 conv if channel sizes differ)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        self.relu_out = nn.ReLU()
        
    def forward(self, x):
        # Main path
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.dropout1(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu2(out)
        out = self.dropout2(out)
        
        # Residual connection
        res = self.residual(x)
        
        return self.relu_out(out + res)


class TCNModel(nn.Module):
    """Temporal Convolutional Network for time series forecasting."""
    
    def __init__(self, input_size, num_channels, kernel_size, output_size, dropout=0.2):
        super(TCNModel, self).__init__()
        
        layers = []
        num_levels = len(num_channels)
        
        for i in range(num_levels):
            dilation = 2 ** i  # Exponentially increasing dilation
            in_ch = input_size if i == 0 else num_channels[i-1]
            out_ch = num_channels[i]
            
            layers.append(TemporalBlock(
                in_ch, out_ch, kernel_size, dilation, dropout
            ))
        
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(num_channels[-1], output_size)
        
    def forward(self, x):
        # x shape: (batch, sequence, features)
        # TCN expects: (batch, channels, sequence)
        x = x.permute(0, 2, 1)
        
        out = self.tcn(x)
        
        # Use last timestep's output
        out = out[:, :, -1]
        out = self.fc(out)
        
        return out

In [12]:
# Create TCN model
tcn_model = TCNModel(
    input_size=input_size,
    num_channels=CONFIG['tcn_num_channels'],
    kernel_size=CONFIG['tcn_kernel_size'],
    output_size=output_size,
    dropout=CONFIG['tcn_dropout']
).to(device)

print(tcn_model)
print(f'\nTotal parameters: {sum(p.numel() for p in tcn_model.parameters()):,}')

TCNModel(
  (tcn): Sequential(
    (0): TemporalBlock(
      (conv1): CausalConv1d(
        (conv): Conv1d(8, 64, kernel_size=(7,), stride=(1,), padding=(6,))
      )
      (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu1): ReLU()
      (dropout1): Dropout(p=0.1, inplace=False)
      (conv2): CausalConv1d(
        (conv): Conv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(6,))
      )
      (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu2): ReLU()
      (dropout2): Dropout(p=0.1, inplace=False)
      (residual): Conv1d(8, 64, kernel_size=(1,), stride=(1,))
      (relu_out): ReLU()
    )
    (1): TemporalBlock(
      (conv1): CausalConv1d(
        (conv): Conv1d(64, 128, kernel_size=(7,), stride=(1,), padding=(12,), dilation=(2,))
      )
      (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu1): ReLU()
      (dropout1): Dropout(p=0.

## 6. Training Functions

In [13]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)


def evaluate(model, data_loader, criterion, device):
    """Evaluate model on validation/test set."""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            predictions = model(batch_x)
            loss = criterion(predictions, batch_y)
            total_loss += loss.item()
    
    return total_loss / len(data_loader)

In [14]:
def train_model(model, train_loader, val_loader, config, model_name, device):
    """Full training loop with early stopping."""
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )
    
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience_counter = 0
    
    print(f'\n{"="*60}')
    print(f'Training {model_name}')
    print(f'{"="*60}')
    
    for epoch in range(config['epochs']):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss = evaluate(model, val_loader, criterion, device)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
        scheduler.step(val_loss)
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # Save best model
            torch.save(model.state_dict(), f'../outputs/models/{model_name.lower()}_best.pth')
        else:
            patience_counter += 1
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch+1:3d}/{config["epochs"]} | '
                  f'Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}')
        
        if patience_counter >= config['early_stopping_patience']:
            print(f'\nEarly stopping at epoch {epoch + 1}')
            break
    
    # Load best model
    model.load_state_dict(torch.load(f'../outputs/models/{model_name.lower()}_best.pth'))
    
    return train_losses, val_losses, best_val_loss

## 7. Train Both Models

In [15]:
# Train LSTM Baseline
lstm_train_losses, lstm_val_losses, lstm_best_val = train_model(
    lstm_model, train_loader, val_loader, CONFIG, 'LSTM', device
)
print(f'\nLSTM Best Validation Loss: {lstm_best_val:.6f}')


Training LSTM
Epoch   1/100 | Train Loss: 0.136719 | Val Loss: 0.046044
Epoch   5/100 | Train Loss: 0.032580 | Val Loss: 0.033295
Epoch  10/100 | Train Loss: 0.029444 | Val Loss: 0.028557
Epoch  15/100 | Train Loss: 0.027423 | Val Loss: 0.027702
Epoch  20/100 | Train Loss: 0.026204 | Val Loss: 0.029200
Epoch  25/100 | Train Loss: 0.024128 | Val Loss: 0.027417
Epoch  30/100 | Train Loss: 0.022804 | Val Loss: 0.027783
Epoch  35/100 | Train Loss: 0.022245 | Val Loss: 0.028357

Early stopping at epoch 38

LSTM Best Validation Loss: 0.027070


In [ ]:
# Train TCN Model
tcn_train_losses, tcn_val_losses, tcn_best_val = train_model(
    tcn_model, train_loader, val_loader, CONFIG, 'TCN', device
)
print(f'\nTCN Best Validation Loss: {tcn_best_val:.6f}')


Training TCN


## 8. Compare Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LSTM Training Curves
axes[0].plot(lstm_train_losses, label='Train Loss', linewidth=2)
axes[0].plot(lstm_val_losses, label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('LSTM Baseline - Training Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# TCN Training Curves
axes[1].plot(tcn_train_losses, label='Train Loss', linewidth=2)
axes[1].plot(tcn_val_losses, label='Validation Loss', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE Loss')
axes[1].set_title('TCN - Training Curves')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Test Set Evaluation

In [ ]:
def compute_metrics(model, data_loader, scaler, device, target_col=0):
    """Compute evaluation metrics on test set."""
    model.eval()
    predictions = []
    actuals = []
    
    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            preds = model(batch_x).cpu().numpy()
            predictions.append(preds)
            actuals.append(batch_y.numpy())
    
    predictions = np.vstack(predictions)
    actuals = np.vstack(actuals)
    
    # Inverse transform to original scale (only for target column)
    mean = scaler.mean_[target_col]
    std = scaler.scale_[target_col]
    
    predictions_orig = predictions * std + mean
    actuals_orig = actuals * std + mean
    
    # Compute metrics
    mse = np.mean((predictions_orig - actuals_orig) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(predictions_orig - actuals_orig))
    
    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'predictions': predictions_orig,
        'actuals': actuals_orig
    }

In [ ]:
# Evaluate both models on test set
lstm_results = compute_metrics(lstm_model, test_loader, scaler, device)
tcn_results = compute_metrics(tcn_model, test_loader, scaler, device)

print('\nTEST SET RESULTS')
print('='*50)
print(f'{"Metric":<10} {"LSTM Baseline":>15} {"TCN":>15}')
print('-'*50)
print(f'{"MSE":<10} {lstm_results["MSE"]:>15.4f} {tcn_results["MSE"]:>15.4f}')
print(f'{"RMSE":<10} {lstm_results["RMSE"]:>15.4f} {tcn_results["RMSE"]:>15.4f}')
print(f'{"MAE":<10} {lstm_results["MAE"]:>15.4f} {tcn_results["MAE"]:>15.4f}')

In [ ]:
# Visualize predictions vs actuals
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot first 500 samples for clarity
n_samples = 500
hours = np.arange(n_samples)

# Average prediction across forecast horizon for visualization
lstm_avg_pred = lstm_results['predictions'][:n_samples].mean(axis=1)
tcn_avg_pred = tcn_results['predictions'][:n_samples].mean(axis=1)
actual_avg = tcn_results['actuals'][:n_samples].mean(axis=1)

# LSTM Predictions
axes[0].plot(hours, actual_avg, label='Actual', color='blue', alpha=0.7, linewidth=1)
axes[0].plot(hours, lstm_avg_pred, label='LSTM Prediction', color='red', alpha=0.7, linewidth=1)
axes[0].set_xlabel('Sample')
axes[0].set_ylabel('Temperature (°C)')
axes[0].set_title('LSTM Baseline - Predictions vs Actual')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# TCN Predictions
axes[1].plot(hours, actual_avg, label='Actual', color='blue', alpha=0.7, linewidth=1)
axes[1].plot(hours, tcn_avg_pred, label='TCN Prediction', color='green', alpha=0.7, linewidth=1)
axes[1].set_xlabel('Sample')
axes[1].set_ylabel('Temperature (°C)')
axes[1].set_title('TCN - Predictions vs Actual')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/predictions_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Summary for Report

In [ ]:
print('='*70)
print('MODEL TRAINING SUMMARY FOR REPORT')
print('='*70)
print(f'''
MODELS TRAINED
--------------
1. LSTM Baseline
   - Hidden Size: {CONFIG['lstm_hidden_size']}
   - Number of Layers: {CONFIG['lstm_num_layers']}
   - Dropout: {CONFIG['lstm_dropout']}
   - Parameters: {sum(p.numel() for p in lstm_model.parameters()):,}

2. Temporal Convolutional Network (TCN)
   - Channels per Layer: {CONFIG['tcn_num_channels']}
   - Kernel Size: {CONFIG['tcn_kernel_size']}
   - Dilation Rates: [1, 2, 4, 8] (exponential)
   - Dropout: {CONFIG['tcn_dropout']}
   - Parameters: {sum(p.numel() for p in tcn_model.parameters()):,}

TRAINING CONFIGURATION
----------------------
Input Window: {CONFIG['sequence_length']} hours (7 days)
Forecast Horizon: {CONFIG['forecast_horizon']} hours (1 day)
Batch Size: {CONFIG['batch_size']}
Learning Rate: {CONFIG['learning_rate']}
Max Epochs: {CONFIG['epochs']}
Early Stopping Patience: {CONFIG['early_stopping_patience']}

TEST SET RESULTS (Temperature Prediction)
------------------------------------------
               LSTM       TCN
MSE:          {lstm_results['MSE']:.4f}    {tcn_results['MSE']:.4f}
RMSE:         {lstm_results['RMSE']:.4f}°C  {tcn_results['RMSE']:.4f}°C
MAE:          {lstm_results['MAE']:.4f}°C   {tcn_results['MAE']:.4f}°C

IMPROVEMENT (TCN vs LSTM)
-------------------------
RMSE Reduction: {((lstm_results['RMSE'] - tcn_results['RMSE']) / lstm_results['RMSE'] * 100):.2f}%
MAE Reduction:  {((lstm_results['MAE'] - tcn_results['MAE']) / lstm_results['MAE'] * 100):.2f}%
''')

---

## Next Steps

1. **Hyperparameter Tuning**: Experiment with different configurations
2. **Ablation Studies**: Test impact of kernel size, dilation rates, etc.
3. **Error Analysis**: Analyze prediction errors by time of day, season
4. **Write Report**: Use these results for Methodology and Results sections